# 학습 모델 평가 · 시각화 노트북

학습해 둔 체크포인트를 불러와 **테스트셋 지표를 재계산**하고, **특정 인덱스의 샘플을 시각화**합니다.
Cursor에서 커널을 `python3.12` 로 선택한 뒤 위에서부터 순서대로 실행하세요.

**사용법**: 아래 *사용자 입력* 셀에서 `TRACK / MODEL / SEQ_LEN / PRED_LEN` 만 바꾸면 됩니다.
- `TRACK` — `uni_a`(PARTICLE, p_gt10, 임계 10) · `uni_b`(XRAY, xrs_long, 임계 1e-5) · `multi`(두 채널 동시)
- 값은 벤치마크 셀과 동일한 설정(`docs/benchmark-conditions.md`)으로 자동 구성되므로 학습 때와 정확히 일치합니다.

**주의**
- 데이터는 `log10` 변환 공간에서 다뤄집니다. 이벤트 임계값도 내부적으로 `log10(임계값)` 으로 비교됩니다.
- GPU를 바꾸려면(또는 CPU↔GPU 전환) **커널을 재시작**하세요 (`CUDA_VISIBLE_DEVICES`는 torch import 전에만 반영됨).


In [ ]:
# ── 사용자 입력 (여기만 수정) ────────────────────────────────────────────
TRACK    = "uni_a"          # uni_a | uni_b | multi
MODEL    = "segrnn_thuml"   # dlinear segrnn_thuml tsmixer patchmixer tide xpatch
                            # patchtst frets itransformer rlinear  (lstm=recursive)
SEQ_LEN  = 288              # 288(1d) · 864(3d) · 2016(7d)
PRED_LEN = 144              # 144(0.5d) · 288(1d)
FOLD     = 0
STRATEGY = "direct"         # direct | recursive(lstm 전용)

SAMPLE_INDICES = [0, 100, 500]   # 시각화할 test 샘플 인덱스
SHOW_PHYSICAL  = False           # True=물리단위(10**x, y로그축) · False=log10 공간
USE_GPU        = True            # False면 CPU
GPU_INDEX      = "1"             # KASI 규칙: GPU 1 고정


In [ ]:
# ── 환경/경로 설정 (torch import 전에 GPU 지정) ──────────────────────────
import os, sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_INDEX if USE_GPU else ""

# 노트북 위치와 무관하게 tslib 가 있는 repo 루트를 찾아 import 경로/작업경로로 설정
REPO = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "tslib").exists() and (p / "main.py").exists():
        REPO = p
        break
assert REPO is not None, "repo 루트(tslib/, main.py)를 찾지 못했습니다. 노트북을 repo 안에 두세요."
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("REPO:", REPO)


In [ ]:
# ── 학습과 동일한 config 재구성 + 체크포인트 확인 ────────────────────────
import torch
from tslib.configs.config import exp_parser, config_postprocess
from tslib.benchmark import driver

cell = {"track": TRACK, "seq_len": SEQ_LEN, "pred_len": PRED_LEN,
        "fold": FOLD, "strategy": STRATEGY}
# 드라이버가 쓰는 것과 완전히 같은 CLI 플래그로 config 생성 → 학습 설정과 일치
argv = driver.cell_argv(cell, epochs=1, models=[MODEL])
config = config_postprocess(exp_parser().parse_args(argv))

run_name  = driver.run_name_for(TRACK, SEQ_LEN, PRED_LEN, FOLD, STRATEGY)
ckpt_file = REPO / "runs" / run_name / "ckpt" / f"{MODEL}.ckpt"

print("run_name :", run_name)
print("ckpt     :", ckpt_file)
print("존재 여부 :", "OK" if ckpt_file.exists() else "없음 (해당 셀을 먼저 학습했는지 확인)")
print("임계값(물리):", config.event_threshold, "| transform:", config.transform)
assert ckpt_file.exists(), f"체크포인트가 없습니다: {ckpt_file}"


In [ ]:
# ── 데이터 로드 (학습 때와 동일한 fold/윈도우) ───────────────────────────
from tslib.data.loader import DataModule

bundle  = DataModule(config).setup()
test_ds = bundle.test_loader.dataset

TARGETS   = list(bundle.target_cols)      # 예: ['p_gt10'] 또는 ['p_gt10','xrs_long']
TARGET_CH = list(bundle.target_indices)   # 전체 입력 채널에서 타깃 채널의 위치
CAD_MIN   = config.cadence_min            # 스텝당 분(minutes)

print(f"input_size(C)={bundle.input_size} · targets={TARGETS} · target_ch={TARGET_CH}")
print(f"test 윈도우 수: {len(test_ds):,}  (seq_len={SEQ_LEN}, pred_len={PRED_LEN})")


In [ ]:
# ── 모델 빌드 + 체크포인트 로드 (test_only_neural 과 동일 절차) ───────────
from tslib.model import build_model
from tslib.exp.lightning_model import ForecastModule
from tslib.exp.metrics import MetricContext

dev = torch.device("cuda" if (USE_GPU and torch.cuda.is_available()) else "cpu")
ctx = MetricContext(thresholds=config.event_threshold,
                    transform=config.transform, target_cols=TARGETS)

model  = build_model(MODEL, config, bundle.input_size, bundle.target_indices,
                     strategy=STRATEGY)
module = ForecastModule(model, config, ctx, strategy=STRATEGY)
state  = torch.load(ckpt_file, map_location="cpu")
module.load_state_dict(state["state_dict"])
module.eval().to(dev)
print(f"'{MODEL}' 로드 완료 · device={dev}")


In [ ]:
# ── 전체 test 예측 수집 + 지표 재계산 ────────────────────────────────────
import numpy as np
import pandas as pd
from tslib.exp.metrics import run_metrics

preds, trues = [], []
with torch.no_grad():
    for x, y in bundle.test_loader:
        p = module(x.to(dev)).cpu().numpy()   # (B, pred_len, T)
        preds.append(p)
        trues.append(y.numpy())               # (B, pred_len, T) — 이미 타깃만
PRED = np.concatenate(preds, 0)   # (N, pred_len, T)
TRUE = np.concatenate(trues, 0)   # (N, pred_len, T)

metrics = run_metrics(PRED, TRUE, ctx, config.metrics)
print("PRED/TRUE:", PRED.shape, TRUE.shape, "| PRED에 NaN:", bool(np.isnan(PRED).any()))
pd.DataFrame([metrics]).T.rename(columns={0: "value"}).round(4)


In [ ]:
# ── 시각화 유틸 ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
%matplotlib inline

def _thr_stored(j):
    thr = float(config.event_threshold[j])
    return np.log10(thr) if config.transform == "log10" else thr

def _phys(v):   # 표시용 변환
    return 10.0 ** v if (SHOW_PHYSICAL and config.transform == "log10") else v

def event_indices(j=0, source="true", n=30):
    \"\"\"타깃 j 에서 이벤트(>=임계값)가 실제로 발생한 test 샘플 인덱스.\"\"\"
    arr = (TRUE if source == "true" else PRED)[:, :, j]
    return np.where((arr >= _thr_stored(j)).any(axis=1))[0][:n]

def visualize(idx):
    \"\"\"test 샘플 하나를 타깃 채널별로: 입력이력·실제·예측·임계값 표시.\"\"\"
    xh = test_ds[idx][0].numpy()          # (seq_len, C)
    yt, yp = TRUE[idx], PRED[idx]         # (pred_len, T)
    T = len(TARGETS)
    fig, axes = plt.subplots(T, 1, figsize=(11, 3.3 * T), squeeze=False)
    t_hist = np.arange(-SEQ_LEN, 0) * CAD_MIN / 60.0
    t_fut  = np.arange(0, PRED_LEN)  * CAD_MIN / 60.0
    unit = "phys" if SHOW_PHYSICAL else "log10"
    for j in range(T):
        ax  = axes[j][0]
        thr = _thr_stored(j)
        true_f, pred_f = yt[:, j], yp[:, j]
        ax.plot(t_hist, _phys(xh[:, TARGET_CH[j]]), color="0.55", lw=1.1, label="history")
        ax.plot(t_fut,  _phys(true_f), color="black",   lw=1.9, label="true")
        ax.plot(t_fut,  _phys(pred_f), color="crimson", lw=1.6, ls="--", label="pred")
        ax.axvline(0, color="0.8", lw=1)
        ax.axhline(_phys(thr), color="green", ls=":", lw=1.4,
                   label=f"thr={config.event_threshold[j]:g}")
        if SHOW_PHYSICAL and config.transform == "log10":
            ax.set_yscale("log")
        ev_t = bool((true_f >= thr).any()); ev_p = bool((pred_f >= thr).any())
        ax.set_title(f"[{MODEL} | {TRACK}] idx={idx} | {TARGETS[j]} | "
                     f"event  true={'O' if ev_t else 'X'}  pred={'O' if ev_p else 'X'}")
        ax.set_xlabel("time (h),  0 = forecast start")
        ax.set_ylabel(f"{TARGETS[j]} ({unit})")
        ax.legend(loc="upper left", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print("실제 이벤트 발생 샘플 예시 (target 0):", event_indices(0)[:15].tolist())


In [ ]:
# ── 지정한 인덱스 시각화 ─────────────────────────────────────────────────
for i in SAMPLE_INDICES:
    if 0 <= i < len(test_ds):
        visualize(i)
    else:
        print(f"idx {i} 범위초과 (test size={len(test_ds)})")


### 팁
- **이벤트 표본만 보고 싶을 때**: `event_indices(0)` 가 돌려주는 인덱스를 `SAMPLE_INDICES` 에 넣고 시각화 셀을 다시 실행하세요. (우주기상 이벤트는 희소해서 임의 인덱스 대부분은 잔잔합니다.)
- **물리단위로 보기**: 입력 셀에서 `SHOW_PHYSICAL = True` → y축 로그스케일 + `10**x` 로 표시.
- **다른 모델/조건 비교**: `MODEL`/`SEQ_LEN`/`PRED_LEN`/`TRACK` 만 바꿔 위에서부터 재실행. (GPU 변경 시에는 커널 재시작)
- `PRED에 NaN: True` 로 나오면 그 체크포인트는 학습이 발산한 것입니다 (예: `xpatch`, seq_len=2016).
